# Shor order finding for 15

Run the compiled period-finding core used in the small N=15 Shor demonstration and recover non-trivial factors classically.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [2]:
from fractions import Fraction
from math import gcd
from qiskit.circuit.library import QFT

# For a=2 mod 15, the modular order is r=4. The phase-register state below is
# the exact compiled order-finding core after modular exponentiation.
counting = 4
circuit = QuantumCircuit(counting)
circuit.h(range(counting))
for wire in range(counting):
    circuit.p(2 * np.pi * (2 ** wire) / 4, wire)
circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))

def probabilities_from_qiskit():
    return Statevector.from_instruction(circuit).probabilities()

reference, reference_ms, _ = benchmark(probabilities_from_qiskit)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = transpile(circuit, backend, optimization_level=1)

def probabilities_from_mettleq():
    state = backend.run(compiled, shots=1, return_statevector=True).result().data(0)["statevector"]
    return np.abs(np.asarray(state)) ** 2

candidate, mettleq_ms, _ = benchmark(probabilities_from_mettleq)
error = max_abs_error(reference, candidate)
phase_integer = int(np.argmax(candidate))
phase_fraction = Fraction(phase_integer, 2 ** counting).limit_denominator(15)
order = phase_fraction.denominator
factors = sorted({gcd(pow(2, order // 2) - 1, 15), gcd(pow(2, order // 2) + 1, 15)})
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/09_shor_order_finding.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="compiled phase distribution atol=2e-6 and factors 3,5",
    passed=error <= 2e-6 and factors == [3, 5],
    exact_match=factors == [3, 5],
    selected_method=method,
    selected_device=device,
    metrics={"max_probability_error": error, "phase_integer": phase_integer, "order": order, "factors": factors},
    notes="This is the small compiled order-finding core, not a scalable modular-arithmetic implementation.",
)

TUTORIAL_RESULT::{"check": "compiled phase distribution atol=2e-6 and factors 3,5", "exact_match": true, "framework": "qiskit", "machine": "arm64", "metrics": {"factors": [3, 5], "max_probability_error": 2.3841857776929487e-07, "order": 4, "phase_integer": 4}, "mettleq_median_ms": 0.6129170069471002, "notebook": "qiskit/09_shor_order_finding.ipynb", "notes": "This is the small compiled order-finding core, not a scalable modular-arithmetic implementation.", "passed": true, "python": "3.13.2", "reference_median_ms": 0.40729198371991515, "reference_over_mettleq": 0.6645140844575518, "schema_version": 1, "selected_device": "cpu", "selected_method": "statevector"}


/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/ipykernel_84145/39546050.py:12: DeprecationWarning: The class ``qiskit.circuit.library.basis_change.qft.QFT`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. ('Use qiskit.circuit.library.QFTGate or qiskit.synthesis.qft.synth_qft_full instead, for access to all previous arguments.',)
  circuit.append(QFT(counting, inverse=True, do_swaps=True).to_gate(), range(counting))
